# NB3B - Map

This notebook processes Pokémon-related data from SQL databases and generates a Pokémon-themed map using Folium.

## Importing Required Libraries

We first import the necessary libraries, including:
- `folium` for interactive map visualization
- `pandas` for data manipulation
- `sqlalchemy` and `sqlite3` for handling databases
- `re` and `unicodedata` for text processing

In [3]:
import folium
import sqlalchemy 
import pandas as pd
import os
import sqlite3
import re
import unicodedata

from folium import FeatureGroup
from folium.plugins import HeatMap, MarkerCluster
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import create_engine, Column, Integer, String, Enum, Float
from folium import IFrame
from branca.colormap import linear

ModuleNotFoundError: No module named 'folium'

## Loading and Merging Pokémon Data  

To analyze and visualize Pokémon locations, we first extract relevant data from two SQLite databases:  

- **`main.db`** → Contains Pokémon rankings, names, and descriptions.  
- **`places.db`** → Stores location data associated with Pokémon.  

### **Steps Involved:**  
1. Establish connections to both databases.  
2. Extract Pokémon rankings, names, and descriptions from `main.db`.  
3. Retrieve location details from `places.db`.  
4. Merge the two datasets based on Pokémon rankings.  

This results in a structured dataset linking Pokémon to specific locations, allowing us to generate meaningful visualizations.  


In [2]:
# Define the directory for the databases
data_directory = "../data/"

# Create engines for both databases with the directory paths
engine_main = create_engine(f"sqlite:///{data_directory}main.db", echo=False)
engine_places = create_engine(f"sqlite:///{data_directory}places.db", echo=False)

# Function to merge pokemon tables based on rank with different table names
def merge_pokemon_tables(main_table_name, places_table_name, engine_main, engine_places):
    # Load the tables from both databases into Pandas
    df_main = pd.read_sql(f"SELECT * FROM {main_table_name}", engine_main)
    df_places = pd.read_sql(f"SELECT * FROM {places_table_name}", engine_places)
    
    # Determine the smaller table size
    min_rows = min(len(df_main), len(df_places))
    
    # Sort by 'rank' and keep only the min_rows (to match the smaller table)
    df_main = df_main.sort_values(by="ranking").head(min_rows)
    df_places = df_places.sort_values(by="ranking").head(min_rows)
    
    # Perform a SQL-style JOIN on 'rank'
    merged_df = pd.merge(df_main, df_places, on="ranking", suffixes=("_main", "_places"))
    
    return merged_df

# Merge the ice_pokemon tables (with different names in main.db and places.db)
merged_ice_pokemon = merge_pokemon_tables("ice_pokemon", "coldest_places", engine_main, engine_places)

# Merge the fire_pokemon tables (with different names in main.db and places.db)
merged_fire_pokemon = merge_pokemon_tables("fire_pokemon", "hottest_places", engine_main, engine_places)

# Merge the water_pokemon tables (with different names in main.db and places.db)
merged_water_pokemon = merge_pokemon_tables("water_pokemon", "wettest_places", engine_main, engine_places)

# Save merged tables into a NEW database inside the same directory
engine_merged = create_engine(f"sqlite:///{data_directory}merged.db", echo=False)

# Save the merged ice_pokemon, fire_pokemon, and water_pokemon tables to the merged database
merged_ice_pokemon.to_sql("ice_pokemon", engine_merged, if_exists="replace", index=False)
merged_fire_pokemon.to_sql("fire_pokemon", engine_merged, if_exists="replace", index=False)
merged_water_pokemon.to_sql("water_pokemon", engine_merged, if_exists="replace", index=False)

print("Merged tables saved to data/merged.db")


Merged tables saved to data/merged.db


## Cleaning and Fixing Pokémon Descriptions  

Pokémon descriptions may contain various formatting issues, such as:  
- Incorrect capitalization (e.g., `POKéMON` → `Pokémon`)  
- Extra spaces or special characters  
- Incorrect possessive forms (e.g., `Trainers` → `Trainer's`)  
- Missing punctuation (e.g., `doesnt` → `doesn't`)  

A custom function is applied to automatically fix these errors, ensuring that all descriptions are properly formatted before they are used in the final dataset.


In [12]:
# Define the grammar fix function
def fix_grammar_in_dataframe(df):
    # Function to fix the grammar of a single Pokémon description
    def fix_grammar(description):
        # Fix "POKéMON" to "Pokémon"
        description = re.sub(r'POKéMON', 'Pokémon', description)
        # Normalize Unicode characters (e.g., handle special characters)
        description = unicodedata.normalize("NFKC", description)
        # Replace double spaces with a single space
        description = re.sub(r'\s{2,}', ' ', description)
        # Fix broken word "con­tinuously" (which might have hidden characters)
        description = re.sub(r'con­tinuously', 'continuously', description)
        # Merge words with a hyphen, such as "X- Y" to "X-Y"
        description = re.sub(r'(\w)- (\w)', r'\1-\2', description)
        # Capitalize all uppercase words
        description = re.sub(r'\b([A-Z]+)\b', lambda match: match.group(0).capitalize(), description)
        # Ensure a space between words if there is a form feed character
        description = re.sub(r'(\w)\f(\w)', r'\1 \2', description)
        # Add space after full stops if missing
        description = re.sub(r'\.(\S)', r'. \1', description)
        # Ensure space after full stop and before form feed character
        description = re.sub(r'\.(\f)', r'. \1', description)
        # Remove unwanted characters (non-alphanumeric, non-whitespace)
        description = re.sub(r'[^\w\s.,;?!\'"-]', '', description)
        # Fix broken word "Un able" to "Unable"
        description = re.sub(r'\bUn\s+able\b', 'Unable', description)
        # Remove non-printable characters
        description = ''.join(char for char in description if char.isprintable())
        # Fix "Its" followed by certain words to "It's"
        description = re.sub(r'\bIts\b(\s+(highly|feeling|apparently|said|not))', r"It's\1", description)
        # Keep "Its" if followed by a noun (indicating possession)
        description = re.sub(r'\bIts\b(\s+[a-zA-Z]+)', lambda m: m.group(0), description)
        # Fix common mistakes like "doesnt" to "doesn't" and "cant" to "can't"
        description = re.sub(r"\bdoesnt\b", "doesn't", description)
        description = re.sub(r"\bcant\b", "can't", description)
        # Standardize "UFO" to uppercase
        description = re.sub(r'\bufo\b', "UFO", description, flags=re.IGNORECASE)
        
        # Ensure possessive "Trainer's" is used correctly
        if "Trainer's" not in description:
            description = re.sub(r'\bTrainers\b', "Trainer's", description)
        
        # Ensure possessive "Pokémon's" is used correctly
        if "Pokémon's" not in description:
            description = re.sub(r'\bPokémons\b', "Pokémon's", description)
        
        # Fix possessive "Skeledirges" to "Skeledirge's"
        description = re.sub(r'\bSkeledirges\b', "Skeledirge's", description)
        # Fix possessive "Regices" to "Regice's"
        description = re.sub(r'\bRegices\b', "Regice's", description)
        # Fix "Pokémonanything" to "Pokémon - anything"
        description = re.sub(r'Pokémonanything', 'Pokémon - anything', description)
        # Fix "gather ing" to "gathering"
        description = re.sub(r' gather ing', ' gathering', description)
        
        return description

    # Apply the grammar fix function to the "pokemon_description" column
    df['pokemon_description'] = df['pokemon_description'].apply(fix_grammar)
    
    return df


In [10]:
#conn = sqlite3.connect('../data/merged.db')

# Query to get data from all three tables
#fire_query = "SELECT * FROM fire_pokemon"
#water_query = "SELECT * FROM water_pokemon"
#ice_query = "SELECT * FROM ice_pokemon"

#proposed code by Jonathan
fire_query = '''
SELECT f.*, h.* 
  FROM fire_pokemon f
  JOIN hottest_places h
    ON f.ranking = h.ranking
'''

water_query = ''' 
SELECT w1.*, w2.*
  FROM water_pokemon w1
  JOIN wettest_places w2
    ON w1.ranking = w2.ranking
'''

ice_query = ''' 
SELECT i.*, c.*
  FROM ice_pokemon i
  JOIN coldest_places c
    ON i.ranking = c.ranking
'''

# Load the data into separate pandas DataFrames
fire_df = pd.read_sql_query(fire_query, conn)
water_df = pd.read_sql_query(water_query, conn)
ice_df = pd.read_sql_query(ice_query, conn)

In [13]:
# Apply the function to clean the descriptions
ice_df = fix_grammar_in_dataframe(ice_df)
fire_df = fix_grammar_in_dataframe(fire_df)
water_df = fix_grammar_in_dataframe(water_df)

# Now save the updated DataFrame back to the SQL database
ice_df.to_sql('ice_pokemon', conn, if_exists='replace', index=False)

water_df.to_sql('water_pokemon', conn, if_exists='replace', index=False)

fire_df.to_sql('fire_pokemon', conn, if_exists='replace', index=False)

# Close the connection
conn.close()

NameError: name 're' is not defined

## Creating an Interactive Pokémon Map  

Using **Folium**, we generate a map that visualizes Pokémon locations based on the merged dataset.  

Features of the map:  
- Markers for each Pokémon’s location.  
- Popup information displaying Pokémon names and descriptions.  
- Different marker colors to distinguish between Pokémon types (e.g., Fire, Water, Ice).  

This interactive visualization helps in understanding the geographical distribution of Pokémon based on their rankings.

## Visualizing Pokémon Distribution on the Map  

After merging Pokémon data with geographical and climate information, we can create a detailed map using **Folium** that displays Pokémon locations across different regions.  

### **Map Features:**  
1. **Pokémon Icons for Locations** – Each Pokémon icon represents a specific Pokémon's observed location and shows their basic stats and description, making it easy to identify their distribution.  
2. **Integrated Heatmap with Climate Data** – The map includes a heatmap layer that visualizes temperature or rainfall intensity across regions, with markers at different regions that can be clicked on to display information (names of regions, rainfall/ temperature).  
3. **Temperature and Rainfall Scale** – A scale is provided to interpret climate variations, helping to correlate Pokémon distribution with environmental conditions.  
4. **Toggle Layers for Custom Views** – Users can switch between different layers, choosing to display different types of Pokémons (water, fire, ice) and a heatmap of different regions (temperature, rainfall) or a combination of these.  

### **Key Insights from the Map:**  
- The distribution of Water-type Pokémon aligns with areas experiencing high rainfall.  
- Fire-type Pokémon tend to appear in warmer regions, while Ice-type Pokémon are more common in colder climates.  
- Comparing Pokémon distribution with environmental data can help identify trends in their habitat preferences. 

This visualization provides a dynamic way to analyze Pokémon distribution in relation to environmental factors, enhancing our understanding of their habitats.  


In [6]:
# Define database path
DATABASE_URL = "sqlite:///../data/merged.db"

# Connect to the database
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)
session = Session()

# Define ORM base
Base = declarative_base()

# Define ORM model for Fire Pokémon
class FirePokemon(Base):
    __tablename__ = "fire_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    hp_stat = Column(Float)
    attack_stat = Column(Float)
    defense_stat = Column(Float)
    special_attack_stat = Column(Float)
    special_defense_stat = Column(Float)
    speed_stat = Column(Float)
    ranking = Column(Float)

# Define ORM model for Ice Pokémon
class IcePokemon(Base):
    __tablename__ = "ice_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    hp_stat = Column(Float)  # Added stats
    attack_stat = Column(Float)
    defense_stat = Column(Float)
    special_attack_stat = Column(Float)
    special_defense_stat = Column(Float)
    speed_stat = Column(Float)
    ranking = Column(Float)

# Define ORM model for Water Pokémon
class WaterPokemon(Base):
    __tablename__ = "water_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    hp_stat = Column(Float)  # Added stats
    attack_stat = Column(Float)
    defense_stat = Column(Float)
    special_attack_stat = Column(Float)
    special_defense_stat = Column(Float)
    speed_stat = Column(Float)
    ranking = Column(Float)


# Fetch Pokémon data
fire_pokemon_entries = session.query(FirePokemon).all()
ice_pokemon_entries = session.query(IcePokemon).all()
water_pokemon_entries = session.query(WaterPokemon).all()  # Fetch Water Pokémon entries

/tmp/ipykernel_5255/746272738.py:10: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [7]:
# Initialize the map
map_center = [20.0, 0.0]  # World center
pokemon_map = folium.Map(location=map_center, zoom_start=2)

# Create MarkerCluster groups for Fire, Ice, and Water Pokémon
fire_cluster = MarkerCluster(name="Fire Pokémon").add_to(pokemon_map)
ice_cluster = MarkerCluster(name="Ice Pokémon").add_to(pokemon_map)
water_cluster = MarkerCluster(name="Water Pokémon").add_to(pokemon_map)  # Add Water Pokémon cluster

def add_pokemon_markers(pokemon_entries, cluster_group):
    for pokemon in pokemon_entries:
        lat, lon = pokemon.latitude, pokemon.longitude  

        # Ensure values exist
        name = pokemon.name if pokemon.name else "Unknown"
        description = pokemon.pokemon_description if pokemon.pokemon_description else "No description available"
        stats = pokemon.total_stat if pokemon.total_stat else "N/A"
        ranking = pokemon.ranking if pokemon.ranking else "Unranked"
        portrait_url = pokemon.pokemon_portrait if pokemon.pokemon_portrait else "https://via.placeholder.com/150"

        # Fetch full stats (ensuring all Pokémon types have full stats)
        hp = f"{pokemon.hp_stat}" if hasattr(pokemon, "hp_stat") and pokemon.hp_stat is not None else "N/A"
        attack = f"{pokemon.attack_stat}" if hasattr(pokemon, "attack_stat") and pokemon.attack_stat is not None else "N/A"
        defense = f"{pokemon.defense_stat}" if hasattr(pokemon, "defense_stat") and pokemon.defense_stat is not None else "N/A"
        sp_attack = f"{pokemon.special_attack_stat}" if hasattr(pokemon, "special_attack_stat") and pokemon.special_attack_stat is not None else "N/A"
        sp_defense = f"{pokemon.special_defense_stat}" if hasattr(pokemon, "special_defense_stat") and pokemon.special_defense_stat is not None else "N/A"
        speed = f"{pokemon.speed_stat}" if hasattr(pokemon, "speed_stat") and pokemon.speed_stat is not None else "N/A"

        tooltip_html = f"""
        <div style="text-align: center; width: 240px; max-width: 240px; padding: 10px; background-color: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); word-wrap: break-word; overflow-wrap: break-word;">
            <h4 style="font-size: 16px; font-weight: bold; color: #D34B29; margin: 0;">{name}</h4>
            <img src="{portrait_url}" width="150px" style="cursor: pointer; border-radius: 8px; max-width: 100%; height: auto;">
            <br><br>
            <b style="font-size: 14px;">Ranking:</b> <span style="font-size: 12px; color: #555;">{ranking}</span><br>
            <b style="font-size: 14px;">Total Stats:</b> <span style="font-size: 12px; color: #555;">{stats}</span><br>
            <hr>
            <b style="font-size: 14px;">HP:</b> <span style="font-size: 12px; color: #555;">{hp}</span><br>
            <b style="font-size: 14px;">Attack:</b> <span style="font-size: 12px; color: #555;">{attack}</span><br>
            <b style="font-size: 14px;">Defense:</b> <span style="font-size: 12px; color: #555;">{defense}</span><br>
            <b style="font-size: 14px;">Sp. Attack:</b> <span style="font-size: 12px; color: #555;">{sp_attack}</span><br>
            <b style="font-size: 14px;">Sp. Defense:</b> <span style="font-size: 12px; color: #555;">{sp_defense}</span><br>
            <b style="font-size: 14px;">Speed:</b> <span style="font-size: 12px; color: #555;">{speed}</span><br>
            <hr>
            <p style="font-size: 12px; text-align: justify; color: #333; margin-top: 8px; line-height: 1.5; word-wrap: break-word; overflow-wrap: break-word; white-space: normal; max-width: 200px; padding: 0; margin: 0;">{description}</p>
        </div>
        """

        tooltip = folium.Tooltip(tooltip_html, sticky=True)  # Keeps tooltip visible on hover

        # Create a DivIcon with the Pokémon image
        icon_html = f"""
        <div style="
            background: url('{portrait_url}') no-repeat center center;
            background-size: contain;
            width: 50px; height: 50px;">
        </div>
        """
        div_icon = folium.DivIcon(html=icon_html)

        # Create a marker with the Pokémon image and tooltip
        marker = folium.Marker(
            location=[lat, lon],
            tooltip=tooltip,
            icon=div_icon  # Use Pokémon image as the marker icon
        )

        # Add marker to the cluster group
        marker.add_to(cluster_group)


# Add Fire Pokémon markers
add_pokemon_markers(fire_pokemon_entries, fire_cluster)

# Add Ice Pokémon markers
add_pokemon_markers(ice_pokemon_entries, ice_cluster)

# Add Water Pokémon markers
add_pokemon_markers(water_pokemon_entries, water_cluster) 

# Add groups to the map
pokemon_map.add_child(fire_cluster)
pokemon_map.add_child(ice_cluster)
pokemon_map.add_child(water_cluster) 

In [8]:
# Define file paths
file_paths = {
    "cold": "/files/ds105a-2024-project-error_105/Temperature exploration/coldest_places.csv",
    "hot": "/files/ds105a-2024-project-error_105/Temperature exploration/hottest_places.csv",
    "wet": "/files/ds105a-2024-project-error_105/Temperature exploration/wettest_places.csv",
}

# Read CSV files
df_places = {key: pd.read_csv(path) for key, path in file_paths.items()}

# Create color scales
colormaps = {
    "temp": linear.YlOrRd_09.scale(df_places["cold"]["temperature"].min(), df_places["hot"]["temperature"].max()),
    "rain": linear.Blues_09.scale(df_places["wet"]["max_rainfall"].min(), df_places["wet"]["max_rainfall"].max()),
}

# Function to create a circle marker with correct units
def add_circle_marker(row, metric, colormap, layer):
    unit = "°C" if metric == "temperature" else "mm"  # Correctly assign units
    popup_content = f"Region: {row.region}<br>{metric.replace('_', ' ').capitalize()}: {row[metric]} {unit}"
    color = colormap(row[metric])

    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=5,
        popup=folium.Popup(popup_content, max_width=300),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(layer)

# Create heatmap layer
heatmap_layer = folium.FeatureGroup(name="HeatMap")

# Apply function to hottest & coldest places (temperature-based)
df_places["hot"].apply(add_circle_marker, axis=1, args=("temperature", colormaps["temp"], heatmap_layer))
df_places["cold"].apply(add_circle_marker, axis=1, args=("temperature", colormaps["temp"], heatmap_layer))

# Apply function to wettest places (rainfall-based)
df_places["wet"].apply(add_circle_marker, axis=1, args=("max_rainfall", colormaps["rain"], heatmap_layer))

# Generate heatmap data
heat_data_temp = [[row.latitude, row.longitude, row.temperature] for _, row in pd.concat([df_places["hot"], df_places["cold"]]).iterrows()]
heat_data_rain = [[row.latitude, row.longitude, row.max_rainfall] for _, row in df_places["wet"].iterrows()]

# Add HeatMap layers
HeatMap(heat_data_temp).add_to(heatmap_layer)
HeatMap(heat_data_rain).add_to(heatmap_layer)

# Add legends and layers to map
colormaps["temp"].caption = 'Temperature Scale (°C)'
colormaps["rain"].caption = 'Rainfall Scale (mm)'
colormaps["temp"].add_to(pokemon_map)
colormaps["rain"].add_to(pokemon_map)

pokemon_map.add_child(heatmap_layer)

In [9]:
folium.LayerControl().add_to(pokemon_map)

# Save the map to an HTML file
pokemon_map.save("pokemon_map.html")

print("Map generated: pokemon_map.html")

session.close()

Map generated: pokemon_map.html
